In [ ]:
from torch.nn import *
from IPython.display import clear_output
from flash_attn import flash_attn_func, flash_attn_kvpacked_func, flash_attn_qkvpacked_func
import torch
import numpy as np
import matplotlib.pyplot as plt
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class Add(Sequential):
    def forward(s, _):
        return sum(s(_)for s in s)
class T(Sequential):
    def forward(s, _):
        return super().forward(_.mT).mT
class RoPE(Module):
    _ = {}
    def forward(s, x):
        _ = s._.get(x.shape[-2:])
        if _ is None:
            _ = torch.arange(x.size(-2),dtype=x.dtype,device=x.device)[:,None]*torch.logspace(0,-4,x.size(-1)//2,dtype=x.dtype,device=x.device)
            _ = _.sin(),_.cos_()
            s._[x.shape[-2:]] = _
        x = x[...,:x.size(-1)//2],x[...,x.size(-1)//2:]
        return torch.cat((x[0]*_[0]-x[1]*_[1], x[0]*_[1]+x[1]*_[0]), -1)
class Attn(Linear):
    def __init__(s, i, h=0, H=0, G=1, g=1, r=4, imp='fa'):
        super().__init__(i, G*g*(h or i)+G*h+G*H)
        s.__dict__ |= locals()
    def forward(s, x):
        _ = super().forward(x)
        if s.imp=='fa' and s.h==s.H and s.g==1 and s.h:
            return flash_attn_qkvpacked_func(_.unflatten(-1,(3,s.G,-1)),window_size=(s.r//2,s.r//2)).flatten(-2)
        q = _[...,:s.G*s.g*(s.h or s.i)].unflatten(-1,(s.G,s.g,-1))
        if s.imp=='fa' and s.h==s.H:
            return flash_attn_kvpacked_func(
                q.flatten(-3,-2),
                _[...,s.G*s.g*s.h:].unflatten(-1,(2,s.g,-1))if s.h else x[...,None,None,:].expand((*x.shape[:-1],2,1,s.i)),
                window_size=(s.r//2,s.r//2)
            ).flatten(-2)
        k = _[...,s.G*s.g*s.h:-s.G*s.H].unflatten(-1,(s.G,-1)) if s.h else x[...,None,:]
        v = _[...,-s.G*s.H:].unflatten(-1,(s.G,-1)) if s.H else x[...,None,:]
        if s.imp=='fa':
            return flash_attn_func(q.flatten(-3,-2),k,v,window_size=(s.r//2,s.r//2)).flatten(-2)
        if s.imp=='sdpa':
            return functional.scaled_dot_product_attention(q.flatten(-3,-2).transpose(-3,-2),k.transpose(-3,-2),v.transpose(-3,-2),enable_gqa=True).transpose(-3,-2).flatten(-2)
        if s.r<=0:
            return ((q @ k.permute(0,2,3,1)[:,None]).softmax(-1) @ v.permute(0,2,1,3)[:,None]).flatten(-3)
        k = functional.pad(k,(0,0,0,0,s.r//2,(s.r+1)//2))
        v = k if s.h==s.H==0 else functional.pad(v,(0,0,0,0,s.r//2,(s.r+1)//2))
        k = k.as_strided(q.shape[:-2]+(k.size(-1),s.r),k.stride()+(k.stride(-3),))
        v = v.as_strided(q.shape[:-2]+(s.r,v.size(-1)),v.stride()[:-1]+(v.stride(-3),v.stride(-1)))
        return ((q @ k).softmax(-1) @ v).flatten(-3)
class SwiGLU(Linear):
    def __init__(s, i, h):
        super().__init__(i, h)
        s._ = Linear(i, h)
    def forward(s, _):
        return functional.silu(super().forward(_)) * s._(_)
def M(i, *h0r0hHGgra):
    m = []
    for h0r0hHGgra in h0r0hHGgra:
        if isinstance(h0r0hHGgra,tuple):
            h0,r0,h,H,G,g,r,a = h0r0hHGgra
            m += Res(
                Add(
                    Sequential(
                        Linear(i, h0, 0),
                        T(Conv1d(h0, h0, r0, padding=r0//2, groups=h0, bias=0)),
                        Linear(h0, (H or i)*G*g, 0) if r else Identity(),
                    ) if r0 else Identity(),
                    *(Sequential(
                        Linear(i, i, 0),
                        RoPE(),
                        Attn(i, h, H, G, g, r),
                    ),) if r else (),
                ),
                LayerNorm((H or i)*G*g if r else h0, elementwise_affine=0),
                SwiGLU((H or i)*G*g if r else h0, a),
                Linear(a, i, 0),
            ),
        else:
            s,h = h0r0hHGgra[:2]
            m += Res(
                Linear(i, h, 0),
                T(MaxPool1d(s)),
                M(*h0r0hHGgra[1:]),
                T(ConvTranspose1d(h, h, s, s, groups=h, bias=0)),
                Linear(h, i, 0),
            ),
    return Sequential(*m)
a = torch.from_numpy(np.memmap('py.bin', mode='r'))
a = a.as_strided_((len(a)//1024-1,2048),(1024,1))
m = Sequential(
    Embedding(92, 64),
    torch.compile(M(64,
        (128,15,0,0,0,0,0,256),
        [2,64,
            (128,15,0,0,1,2,-1,256),
        ],
        (128,15,0,0,0,0,0,256),
    ), fullgraph=True, dynamic=False),
    Linear(64, 92),
).to('cuda',torch.bfloat16)
o = torch.optim.Adam(m.parameters(), 1/1024)

In [ ]:
L,A = [],[]
for y in range(500):
    for y in range(1):
        lc = 0
        for y in range(20):
            for y in range(20):
                t = a[torch.randint(len(a),[16])].to('cuda',torch.long)
                x = t.put(torch.randint(t.numel(),[t.numel()*3//4],device='cuda'),torch.randint(92,[t.numel()*3//4],device='cuda'))
                y = m(x)
                l = functional.cross_entropy(y.mT, t)
                (l*65536).backward()
                o.step()
                o.zero_grad()
            lc += l.data
        L.append(lc[None])
        A.append((y.argmax(-1)==t).sum()[None]/t.numel())
    plt.subplot(121).plot((L:=[torch.cat(L)])[0].to('cpu',torch.float), label=f'{L[0][-1]:.2f}')
    plt.legend()
    plt.subplot(122).plot((A:=[torch.cat(A)])[0].to('cpu',torch.float), label=f'{A[0][-1]:.2f}')
    plt.legend()
    clear_output(True)
    plt.show()